# End-to-End Reproduction Notebook

This notebook allows you to reproduce the project's results using cached data, running the full pipeline from video embedding to analysis without external API costs.

## Steps:
1.  **Setup**: Check and download necessary data.
2.  **Asset Generation**: Generate (or verify) video embeddings.
3.  **Experiment Execution**: Run experiments using the cached LLM responses.
4.  **Analysis**: Visualize and compare results.

## 1. Setup Data
Ensure `disk_cache`, `local`, and `results` are populated.

In [ ]:
import logging
import sys
from pathlib import Path
import os

# Ensure project root is in path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    sys.path.append(str(project_root / "src"))

# Run the download script automatically
!python ../scripts/download_data.py

## 2. Generate Video Embeddings
This step processes videos in `datasets/wild_videos` and saves embeddings to `local/wild_videos_embs`.
Since we downloaded the cache, most of these should already exist and be skipped.

In [ ]:
from data.video_embeddings import VideoEmbedder

# Define paths relative to valid project root
video_dir = project_root / "datasets" / "wild_videos"
output_dir = project_root / "local" / "wild_videos_embs"

embedder = VideoEmbedder()
embedder.process_directory(
    video_dir=video_dir,
    output_dir=output_dir,
    fps=1,
    clip_size=1
)

## 3. Run Experiments
We will run the specified configurations. The `--cached-execution-only` (or `--block-llm`) flag ensures we don't make expensive API calls, but we SHOULD have all responses in the downloaded cache.

In [ ]:
configs_to_run = [
    "config/embs_vs_llms/wild_dev_sim.yaml",
    "config/embs_vs_llms/wild_dev_sim_one_shot_t=1.yaml",
    "config/embs_vs_llms/wild_dev_sim_vec.yaml",
    "config/embs_vs_llms/wild_dev_sim_vec_vid.yaml"
]

for conf in configs_to_run:
    print(f"\n{'='*50}\nRunning: {conf}\n{'='*50}")
    !python ../src/main.py {conf} --block-llm

## 4. Analysis
Now we analyze the results using the `ranking3` logic.
Select two methods from the dropdowns below to compare them.

In [ ]:
import ipywidgets as widgets
from analysis.llm_based import load_dfs, AnalysisArgs
from analysis import ranking3
import matplotlib.pyplot as plt
from IPython.display import display

# Load data
df, df_z = load_dfs("../results/upload/")
methods = sorted(df['method'].unique())

# Create widgets
method1_dropdown = widgets.Dropdown(options=methods, description='Method 1:')
method2_dropdown = widgets.Dropdown(options=methods, description='Method 2:')
metric_text = widgets.Text(value='cos_sim_mean', description='Metric:')
btn = widgets.Button(description="Run Analysis")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        m1 = method1_dropdown.value
        m2 = method2_dropdown.value
        metric = metric_text.value
        
        print(f"Comparing {m1} vs {m2} on {metric}...")
        
        args = AnalysisArgs(
            method1=m1,
            method2=m2,
            metric=metric
        )
        
        # Run the main analysis logic manually to control display
        # Note: ranking3.main saves files, but we want to see them here too.
        ranking3.main(args)
        
        # Since the scripts save plots to files, let's load and display the key comparison plot for num_masked=6
        result_img_path = Path("../results/plots/rank_stability_Nov") / "rank_comparison_at_6_masked.png"
        if result_img_path.exists():
             img = plt.imread(str(result_img_path))
             plt.figure(figsize=(20, 10))
             plt.imshow(img)
             plt.axis('off')
             plt.show()

btn.on_click(on_click)
display(method1_dropdown, method2_dropdown, metric_text, btn, output)